# 01 - Engine check

Confirms the DSSAT engine is installed and that xdssat's writers reproduce a
real rice experiment, then runs a small synthetic grid.

* Engine: built by `python scripts/build_engine.py` (location: `config.ENGINE_DIR`).
* Fixtures: the sibling `dssat-csm-data` repository.
* Template: `Rice/IRMZ8601.RIX` (IRRI Munoz 1986, transplanted IR 58, puddled
  and bunded field, flood depth schedule, split urea).

In [ ]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path
import numpy as np
import pandas as pd

import config
from filex import FileX
from runner import run_units
from sol import read_sol, write_sol
from wth import read_wth, write_wth

DATA = Path('../../dssat-csm-data')
print('engine:', config.engine_exe())
print('work root:', config.WORK_ROOT)

## Template experiment

The template's treatment 5 (140 kg N/ha, continuously flooded) is copied to
every grid-cell season. All its dates move with the planting date.

In [ ]:
template = FileX.read(DATA / 'Rice' / 'IRMZ8601.RIX')
template.treatments()

## A synthetic grid

Weather: the 1986 Munoz year repeated over 1999-2002 with rainfall scaled per
cell. Soil: one IRRI profile. Three seasons a year, planting dates shifted a
little per cell. This only exercises the machinery; real inputs (AgERA5,
rice-field mask, soil grid) come in the next notebooks.

In [ ]:
station, w86 = read_wth(DATA / 'Weather' / 'IRMZ8601.WTH')
days = pd.date_range('1999-01-01', '2002-12-31', freq='D')
base = w86.reset_index(drop=True)
weather = base.iloc[np.minimum(days.dayofyear.values - 1, 364)].set_index(days)
soils = read_sol(DATA / 'Soil' / 'SOIL.SOL', ids=['IBRI910002'])

rows = []
for cell in range(40):
    for year in (2000, 2001):
        for doy in (40, 160, 280):
            rows.append(dict(cell=cell, wsta=f'GR{cell:06d}', soil_id='IBRI910002',
                             lat=15.7, lon=120.9, trno_template=5,
                             pdate=pd.Timestamp(year, 1, 1) + pd.Timedelta(days=doy + cell // 4)))
units = pd.DataFrame(rows)

def write_inputs(run_dir, u):
    for c in u.drop_duplicates('cell').itertuples():
        write_wth(run_dir / f'{c.wsta}.WTH', weather.assign(RAIN=weather.RAIN * (0.6 + c.cell / 50)),
                  'GRID', c.lat, c.lon)
    write_sol(run_dir / 'SOIL.SOL', [soils[s] for s in sorted(set(u.soil_id))])

len(units)

In [ ]:
results = run_units(units, template, write_inputs, chunk_size=100, workers=4,
                    soils=soils, ic_water=0.9)
summary = pd.concat([r.summary for r in results], ignore_index=True)
failed = [r.failed for r in results if len(r.failed)]
print(len(summary), 'seasons simulated,', sum(len(f) for f in failed), 'failed')
summary[['cell', 'pdate', 'PDAT', 'ADAT', 'MDAT', 'HWAH', 'PRCP']].head()

In [ ]:
summary['HWAH'] = summary['HWAH'].astype(float)
summary['season'] = pd.to_datetime(summary['pdate']).dt.month.map(
    lambda m: 'S1' if m < 4 else ('S2' if m < 8 else 'S3'))
summary.groupby('season')['HWAH'].describe().round(0)